In [1]:
from google.colab import drive

In [3]:
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [4]:
import os

In [5]:
os.chdir('/content/drive/MyDrive/ANN')

In [6]:
import pandas as pd
import numpy as np

In [7]:
df = pd.read_csv('imbalanced_data.csv')

In [8]:
df.head()

In [9]:
df['tweet']

In [10]:
import seaborn as sns

In [11]:
import matplotlib.pyplot as plt
sns.countplot(x=df['label'])
plt.title("Class Distribution")
plt.show()

print(df['label'].value_counts(normalize=True))

In [12]:
#Tweet Length Analysis
df['length'] = df['tweet'].apply(len)

plt.hist(df['length'], bins=50)
plt.title("Tweet Length Distribution")
plt.show()

In [13]:
#Most Common Words
from collections import Counter

all_words = " ".join(df['tweet']).split()
common_words = Counter(all_words).most_common(20)

words = [w[0] for w in common_words]
counts = [w[1] for w in common_words]

plt.bar(words, counts)
plt.xticks(rotation=45)
plt.title("Top 20 Words")
plt.show()

In [14]:
sns.boxplot(x='label', y='length', data=df)
plt.title("Tweet Length vs Class")
plt.show()

In [15]:
#WORD CLOUD (HATE vs NON-HATE)
from wordcloud import WordCloud

hate_words = " ".join(df[df['label'] == 1]['tweet'])
non_hate_words = " ".join(df[df['label'] == 0]['tweet'])

# Hate Speech
plt.figure(figsize=(10,5))
wc = WordCloud(background_color='black').generate(hate_words)
plt.imshow(wc)
plt.axis('off')
plt.title("Hate Speech WordCloud")
plt.show()

# Non-Hate Speech
plt.figure(figsize=(10,5))
wc = WordCloud(background_color='white').generate(non_hate_words)
plt.imshow(wc)
plt.axis('off')
plt.title("Non-Hate Speech WordCloud")
plt.show()

In [16]:
from nltk.corpus import stopwords
import re
import string
import nltk
nltk.download('stopwords')


In [17]:
stemmer = nltk.SnowballStemmer("english")
stopword = set(stopwords.words('english'))

In [18]:
def data_cleaning(words):
    words = words.lower()
    words = re.sub(r'@\w+', '', words)  # remove mentions
    words = re.sub(r'http\S+', '', words)  # remove URLs
    words = re.sub(r'[^a-z\s]', '', words)  # remove special chars & numbers
    words = re.sub(r'\s+', ' ', words).strip()

    words = [word for word in words.split() if word not in stopword]
    words=" ".join(words)
    words = [stemmer.stem(word) for word in words.split()]
    words=" ".join(words)

    return words

In [19]:
df.head()

In [20]:
df['tweet']=df['tweet'].apply(data_cleaning)

In [21]:
df.head()

In [22]:
df.iloc[0]['tweet']

In [23]:
X = df['tweet']
y = df['label']

In [24]:
from sklearn.model_selection import train_test_split

In [25]:
X_train,X_test, y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42)

print(len(X_train),len(X_test))
print(len(y_train),len(y_test))

In [32]:
from keras import Sequential
from keras.layers import Dense,GRU, Embedding, Flatten

In [33]:
all_words = [word for sentence in df['tweet'] for word in sentence.split()]

unique_words = set(all_words)
vocab_size = len(unique_words)

print('vocabulary size',vocab_size)

In [35]:
from tensorflow.keras.preprocessing.text import Tokenizer
max_words = 50000
max_len = 300

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [36]:
from tensorflow.keras import Input

model = Sequential()

model.add(Input(shape=(max_len,)))  # 🔥 important

model.add(Embedding(max_words, 100))

model.add(GRU(100, dropout = 0.2,recurrent_dropout=0.2))

model.add(Dense(1, activation='sigmoid'))

model.summary()

In [37]:
from keras.optimizers import RMSprop

In [38]:
model.compile(loss='binary_crossentropy',optimizer=RMSprop(),metrics=['accuracy'])

In [39]:
history = model.fit(
    X_train_pad, y_train,
    batch_size=128,
    epochs=10,
    validation_data=(X_test_pad, y_test)
)

In [40]:
loss, acc = model.evaluate(X_test_pad, y_test)
print("Test Accuracy:", acc)

In [41]:
rnn_prediction = model.predict(X_test_pad)

200/200 ━━━━━━━━━━━━━━━━━━━━ 39s 195ms/step


In [42]:
rnn_prediction

array([[4.1933126e-05],
       [7.2964322e-05],
       [2.8529446e-05],
       ...,
       [1.7967572e-03],
       [1.6265000e-04],
       [9.9265265e-01]], dtype=float32)

In [43]:
res = []
for prediction in rnn_prediction:
    if prediction[0] < 0.5:
        res.append(0)
    else:
        res.append(1)

In [44]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test,res))

[[5767  170]
 [ 147  309]]


In [45]:
import pickle
with open('tokenizeres.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [46]:
import keras

In [47]:
model.save("model2.h5")


In [48]:
load_model=keras.models.load_model("model2.h5")

In [49]:
with open('tokenizeres.pickle', 'rb') as handle:
    load_tokenizer = pickle.load(handle)